# Gym Workout Recommender

Select a Python kernel and **Run All**. Use the interface below to enter your profile, build a personalized workout, and record feedback. Keep this notebook alongside `data/`. All system code is included here; the application Python files are not required.


In [ ]:
# First use only: uncomment this line if your kernel is missing packages, then restart the kernel.
# %pip install numpy pandas scikit-learn ipywidgets ipython


In [ ]:
"""Chapter-aligned recommendation algorithms with common hard constraints."""
from dataclasses import dataclass, field
from pathlib import Path
import math
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT=next((p for p in [Path.cwd(),Path.cwd()/'gym_workout_system'] if (p/'data/processed/exercises.csv').exists()),Path.cwd())
LEVELS={'beginner':0,'intermediate':1,'expert':2}
GOALS=['Beginner','Weight Loss','Muscle Gain','Strength']
METHODS=['Hybrid','Popularity','Content cosine','SVD','User CF','Item CF','Knowledge','Context']
GOAL_TEXT={'Beginner':'beginner strength bodyweight compound',
           'Weight Loss':'cardio endurance aerobic circuit full body',
           'Muscle Gain':'strength hypertrophy muscle compound isolation',
           'Strength':'strength powerlifting compound barbell'}

@dataclass
class Profile:
    user_id: str='local-user'
    age: int=25
    gender: str='Prefer not to say'
    height_cm: float=170
    weight_kg: float=70
    goal: str='Beginner'
    experience: str='beginner'
    equipment: list=field(default_factory=lambda:['bodyweight','dumbbell','bench'])
    muscles: list=field(default_factory=list)
    excluded_muscles: list=field(default_factory=list)
    minutes: int=30
    location: str='Gym'
    energy: str='Normal'

    @property
    def bmi(self): return round(self.weight_kg/(self.height_cm/100)**2,1)

    def validate(self):
        if not self.user_id.strip() or len(self.user_id)>80: raise ValueError('Enter a user ID of 1–80 characters.')
        if self.goal not in GOALS or self.experience not in LEVELS: raise ValueError('Invalid goal or experience.')
        values=[self.age,self.height_cm,self.weight_kg,self.minutes]
        if not all(math.isfinite(float(v)) for v in values): raise ValueError('Profile values must be finite.')
        if not (18<=self.age<=100 and 100<=self.height_cm<=250 and 30<=self.weight_kg<=300 and 10<=self.minutes<=180):
            raise ValueError('Profile values are outside supported ranges.')
        if self.location not in ['Gym','Home','Outdoors'] or self.energy not in ['Low','Normal','High']:
            raise ValueError('Invalid context.')

class Recommender:
    def __init__(self, exercises=None, history=None):
        self.items=(pd.read_csv(ROOT/'data/processed/exercises.csv').fillna('') if exercises is None else exercises.copy()).reset_index(drop=True)
        self.history=history if history is not None else pd.DataFrame(columns=['user_id','exercise_id','rating','completion'])
        text=self.items[['name','description','category','muscle','level','equipment']].agg(' '.join,axis=1)
        self.vectorizer=TfidfVectorizer(stop_words='english',ngram_range=(1,2),min_df=1)
        self.features=self.vectorizer.fit_transform(text)
        self.ids=self.items.exercise_id.tolist()
        self.lookup={eid:i for i,eid in enumerate(self.ids)}

    def collaborative(self,user_id):
        """Observed overlap for CF; centered truncated SVD only when feedback exists.

        Missing centered entries are zero residuals, not zero-star observations.
        This is an educational imputation baseline, not observed-only optimization.
        """
        n=len(self.items)
        zero=np.zeros(n)
        hist=self.history[self.history.exercise_id.isin(self.ids)]
        if hist.empty: return {'SVD':zero,'User CF':zero,'Item CF':zero},False
        matrix=hist.pivot_table(index='user_id',columns='exercise_id',values='rating',aggfunc='mean').reindex(columns=self.ids)
        if user_id not in matrix.index or len(matrix)<2: return {'SVD':zero,'User CF':zero,'Item CF':zero},False
        a=matrix.to_numpy(dtype=float); observed=np.isfinite(a)
        means=np.nanmean(a,axis=1); centered=np.where(observed,a-means[:,None],0)
        uidx=matrix.index.get_loc(user_id)
        if observed[uidx].sum()<2: return {'SVD':zero,'User CF':zero,'Item CF':zero},False
        # Cosine on observed ratings; overlap shrinkage prevents one shared rating dominating.
        filled=np.nan_to_num(a)
        usim=cosine_similarity(filled[uidx:uidx+1],filled)[0]
        overlaps=observed.astype(float)@observed[uidx].astype(float)
        usim*=overlaps/(overlaps+3); usim[uidx]=0
        denom=usim@observed
        user_pred=np.divide(usim@filled,denom,out=np.zeros(n),where=denom>0)/5
        # Compute only columns the user rated, avoiding an N x N item matrix.
        rated=np.flatnonzero(observed[uidx])
        isim=cosine_similarity(filled.T,filled[:,rated].T)
        overlap=observed.T.astype(float)@observed[:,rated].astype(float)
        isim*=overlap/(overlap+3)
        for col,idx in enumerate(rated): isim[idx,col]=0
        item_pred=np.divide(isim@a[uidx,rated],isim.sum(axis=1),out=np.zeros(n),where=isim.sum(axis=1)>0)/5
        u,s,vt=np.linalg.svd(centered,full_matrices=False)
        k=min(8,max(1,len(s)-1))
        svd=np.clip(means[uidx]+(u[uidx,:k]*s[:k])@vt[:k],1,5)/5
        svd[~observed.any(axis=0)]=0
        return {'SVD':svd,'User CF':user_pred,'Item CF':item_pred},True

    def recommend(self,profile,method='Hybrid',top_n=8,exclude_seen=False):
        profile.validate()
        if method not in METHODS: raise ValueError('Unknown recommendation method.')
        if not isinstance(top_n,int) or not 1<=top_n<=100: raise ValueError('top_n must be 1–100.')
        df=self.items.copy()
        equipment=set(profile.equipment)|{'bodyweight'}
        if profile.location=='Outdoors': equipment-= {'machine','cable','smith machine'}
        limit=min(LEVELS[profile.experience],0 if profile.goal=='Beginner' or profile.energy=='Low' else 2)
        allowed=df.level.map(LEVELS).fillna(99)<=limit
        allowed &= df.required_equipment.map(lambda s:set(s.split('|')).issubset(equipment))
        allowed &= ~df.muscle.isin(profile.excluded_muscles)
        if profile.muscles: allowed &= df.muscle.isin(profile.muscles)
        seen=self.history[self.history.user_id.eq(profile.user_id)]
        if exclude_seen: allowed &= ~df.exercise_id.isin(seen.exercise_id)
        query=GOAL_TEXT[profile.goal]+' '+' '.join(profile.muscles)
        q=self.vectorizer.transform([query])
        content=cosine_similarity(q,self.features)[0]
        positives=seen[pd.to_numeric(seen.rating,errors='coerce')>=4]
        indices=[self.lookup[i] for i in positives.exercise_id if i in self.lookup]
        if indices:
            content=.6*content+.4*cosine_similarity(np.asarray(self.features[indices].mean(axis=0)),self.features)[0]
        ratings=pd.to_numeric(df.source_rating,errors='coerce')
        prior=ratings.fillna(ratings.median() if ratings.notna().any() else 5).to_numpy()/10
        # Bayesian shrinkage on genuine app ratings, using the source score as prior.
        agg=self.history.groupby('exercise_id').rating.agg(['mean','count']) if not self.history.empty else pd.DataFrame(columns=['mean','count'])
        counts=df.exercise_id.map(agg['count']).fillna(0).to_numpy()
        averages=df.exercise_id.map(agg['mean']).fillna(0).to_numpy()/5
        popularity=(5*prior+counts*averages)/(5+counts)
        preferred={'Weight Loss':['cardio','plyometrics'],'Strength':['strength','powerlifting'],
                   'Muscle Gain':['strength'],'Beginner':['strength','cardio','stretching']}[profile.goal]
        knowledge=.7*df.category.isin(preferred).to_numpy()+.3*df.level.eq(profile.experience).to_numpy()
        # Local feedback completion supplies a contextual preference signal.
        completion=seen.groupby('exercise_id').completion.mean() if not seen.empty else pd.Series(dtype=float)
        context=.6*knowledge+.4*df.exercise_id.map(completion).fillna(.5).to_numpy()
        cf,ready=self.collaborative(profile.user_id)
        scores={'Popularity':popularity,'Content cosine':content,'Knowledge':knowledge,'Context':context,**cf}
        scores['Hybrid']=.40*content+.20*popularity+.25*knowledge+.15*context
        if ready: scores['Hybrid']=.8*scores['Hybrid']+.2*(cf['SVD']+cf['User CF']+cf['Item CF'])/3
        fallback=method in cf and not ready
        df['score']=scores['Hybrid'] if fallback else scores[method]
        for name,score in scores.items(): df[name]=score
        df=df[allowed].sort_values(['score','name'],ascending=[False,True])
        # Greedy diversity avoids filling a full-body session with one muscle group.
        selected=[]; muscle_counts={}
        while len(df) and len(selected)<top_n:
            adjusted=df.score-df.muscle.map(lambda m:.10*muscle_counts.get(m,0))
            idx=adjusted.idxmax(); row=df.loc[idx].copy(); selected.append(row)
            muscle_counts[row.muscle]=muscle_counts.get(row.muscle,0)+1
            df=df.drop(idx)
        result=pd.DataFrame(selected) if selected else df
        result['reason']=[f'{r.muscle}; {r.level}; available equipment; {profile.goal.lower()} match. '
                          f'Content {r["Content cosine"]:.2f}, prior/feedback {r.Popularity:.2f}, rules {r.Knowledge:.2f}.'
                          for _,r in result.iterrows()]
        return result.reset_index(drop=True), {'eligible':int(allowed.sum()),'collaborative_ready':ready,
             'fallback':fallback,'message':'Hybrid fallback: collaborative methods need this user to rate at least 2 exercises and at least 2 users overall.' if fallback else ''}

def make_plan(ranked,profile):
    """Transparent demo time allocation; durations include sets/rest/transitions."""
    if ranked.empty: return ranked.copy()
    profile.validate()
    block=min(5 if profile.energy=='Low' else 7,profile.minutes-5)
    count=min(len(ranked),max(0,(profile.minutes-5)//block))
    plan=ranked.head(count).copy()
    plan['duration_minutes']=block
    plan['suggested_format']=plan.category.map(lambda c:'Easy timed practice' if c in ['cardio','stretching'] else 'Technique-focused sets; choose comfortable load')
    return plan

def comparable_sessions(profile,members,k=20):
    """Case-based retrieval from source sessions; never creates exercise-level labels."""
    cols=['Age','Weight (kg)','Height (m)','Experience_Level']
    target=np.array([profile.age,profile.weight_kg,profile.height_cm/100,LEVELS[profile.experience]+1])
    values=members[cols].apply(pd.to_numeric,errors='coerce')
    valid=values.notna().all(axis=1)
    scale=values[valid].std().replace(0,1)
    distance=((values[valid]-target)/scale).pow(2).mean(axis=1).pow(.5)
    cases=members.loc[distance.nsmallest(k).index].copy()
    cases['similarity']=1/(1+distance.loc[cases.index])
    return cases


"""Local SQLite profiles and genuine feedback; no fabricated training interactions."""
from pathlib import Path
import json
import sqlite3
from datetime import datetime, timezone
import pandas as pd

DB = ROOT / 'data/notebook_workouts.sqlite3'

def connect(path=DB):
    con=sqlite3.connect(path)
    con.execute('PRAGMA foreign_keys = ON')
    con.executescript('''
    CREATE TABLE IF NOT EXISTS profiles(user_id TEXT PRIMARY KEY, profile_json TEXT NOT NULL);
    CREATE TABLE IF NOT EXISTS activities(
      activity_id INTEGER PRIMARY KEY AUTOINCREMENT,
      user_id TEXT NOT NULL REFERENCES profiles(user_id), exercise_id TEXT NOT NULL,
      duration REAL NOT NULL CHECK(duration>0), completion REAL NOT NULL CHECK(completion BETWEEN 0 AND 1),
      rating REAL NOT NULL CHECK(rating BETWEEN 1 AND 5), created_at TEXT NOT NULL);
    ''')
    return con

def save_profile(profile, path=DB):
    profile.validate()
    from dataclasses import asdict
    with connect(path) as con:
        con.execute('INSERT INTO profiles VALUES(?,?) ON CONFLICT(user_id) DO UPDATE SET profile_json=excluded.profile_json',
                    (profile.user_id,json.dumps(asdict(profile))))

def profiles(path=DB):
    with connect(path) as con:
        return {uid:json.loads(data) for uid,data in con.execute('SELECT * FROM profiles ORDER BY user_id')}

def log_activity(user_id, exercise_id, duration, completion, rating, path=DB):
    import math
    if not all(math.isfinite(float(v)) for v in [duration,completion,rating]):
        raise ValueError('Activity values must be finite.')
    if not 0 < duration <= 480 or not 0 <= completion <= 1 or not 1 <= rating <= 5:
        raise ValueError('Invalid duration, completion, or rating.')
    with connect(path) as con:
        con.execute('INSERT INTO activities(user_id,exercise_id,duration,completion,rating,created_at) VALUES(?,?,?,?,?,?)',
                    (user_id,exercise_id,duration,completion,rating,datetime.now(timezone.utc).isoformat()))

def activities(path=DB):
    with connect(path) as con:
        return pd.read_sql_query('SELECT * FROM activities ORDER BY created_at,activity_id',con)


In [ ]:

import html
import base64
import ipywidgets as w
from IPython.display import display, HTML, clear_output

catalogue = pd.read_csv(ROOT/'data/processed/exercises.csv').fillna('')
exercise_names = dict(zip(catalogue.exercise_id, catalogue.name))
state = {'profile': None, 'plan': pd.DataFrame(), 'recommendations': pd.DataFrame()}
style = {'description_width': '130px'}
wide = w.Layout(width='440px')

def field(widget, **kwargs):
    return widget(style=style, layout=wide, **kwargs)

def escape(value):
    return html.escape(str(value))

def download_link(label, text, filename, mime='text/csv'):
    encoded = base64.b64encode(text.encode('utf-8')).decode('ascii')
    return f'<a download="{escape(filename)}" href="data:{mime};base64,{encoded}">{escape(label)}</a>'

saved_profiles = field(w.Dropdown, description='Saved profile', options=['New profile']+list(profiles()))
user_id = field(w.Text, description='Profile name', value='my-profile')
goal = field(w.Dropdown, description='Goal', options=GOALS, value='Muscle Gain')
experience = field(w.Dropdown, description='Experience', options=list(LEVELS), value='beginner')
age = field(w.BoundedIntText, description='Age', min=18, max=100, value=25)
gender = field(w.Dropdown, description='Gender', options=['Prefer not to say','Female','Male','Other'])
height = field(w.BoundedFloatText, description='Height (cm)', min=100, max=250, value=170)
weight = field(w.BoundedFloatText, description='Weight (kg)', min=30, max=300, value=70)
available = sorted(set('|'.join(catalogue.required_equipment).split('|'))-{'unknown','other','bodyweight'})
equipment = field(w.SelectMultiple, description='Equipment', options=available, value=('dumbbell','bench'), rows=7)
focus = field(w.SelectMultiple, description='Focus muscles', options=sorted(catalogue.muscle.unique()), rows=5)
exclude = field(w.SelectMultiple, description='Exclude muscles', options=sorted(catalogue.muscle.unique()), rows=5)
minutes = field(w.IntSlider, description='Minutes available', min=10, max=180, step=5, value=30)
location = field(w.Dropdown, description='Location', options=['Gym','Home','Outdoors'])
energy = field(w.Dropdown, description='Energy today', options=['Low','Normal','High'], value='Normal')
new_only = field(w.Checkbox, description='Only new exercises', value=False)
build_button = w.Button(description='Build my workout', button_style='success', icon='play', layout=w.Layout(width='220px'))
load_button = w.Button(description='Load profile', icon='folder-open')
status = w.HTML()
workout_view, history_view, library_view = w.Output(), w.Output(), w.Output()

exercise = field(w.Dropdown, description='Exercise', options=[(name,eid) for eid,name in exercise_names.items()])
duration = field(w.BoundedFloatText, description='Actual minutes', min=1, max=480, value=7)
completion = field(w.IntSlider, description='Completion (%)', min=0, max=100, value=100)
rating = field(w.IntSlider, description='Your rating', min=1, max=5, value=4)
log_button = w.Button(description='Save workout feedback', button_style='info', icon='check', disabled=True)
feedback_status = w.HTML()
search = field(w.Text, description='Search exercises', placeholder='e.g. squat, row, curl')


def read_profile():
    profile = Profile(user_id=user_id.value.strip(),age=age.value,gender=gender.value,
        height_cm=height.value,weight_kg=weight.value,goal=goal.value,experience=experience.value,
        equipment=['bodyweight']+list(equipment.value),muscles=list(focus.value),
        excluded_muscles=list(exclude.value),minutes=minutes.value,location=location.value,energy=energy.value)
    profile.validate()
    return profile


def render_history():
    with history_view:
        clear_output(wait=True)
        if state['profile'] is None:
            display(HTML('<p>Build or load a profile to view your activity.</p>'))
            return
        own = activities()
        own = own[own.user_id.eq(state['profile'].user_id)]
        if own.empty:
            display(HTML('<p>No activity yet. Record an exercise after completing it.</p>'))
            return
        view = own.merge(catalogue[['exercise_id','name']],on='exercise_id',how='left')
        display(HTML(f'<h3>{len(own)} exercises logged ? {own.duration.sum():g} minutes ? '
                     f'{own.completion.mean():.0%} average completion</h3>'))
        display(view[['created_at','name','duration','completion','rating']].rename(columns={
            'created_at':'Date (UTC)','name':'Exercise','duration':'Minutes','completion':'Completion','rating':'Rating'}))
        display(HTML(download_link('Download my activity',view.to_csv(index=False),'my_activity.csv')))


def render_workout(profile, plan, meta):
    with workout_view:
        clear_output(wait=True)
        display(HTML(f'<h2>{escape(profile.goal)} ? {escape(profile.user_id)}</h2>'
            f'<p>{profile.minutes} minutes available ? {escape(profile.location)} ? '
            f'{escape(profile.experience.title())} ? {meta["eligible"]} matching exercises</p>'))
        if plan.empty:
            display(HTML('<p><b>No workout fits these settings.</b> Adjust equipment or muscle selections and build again.</p>'))
            return
        display(HTML(f'<p><b>{len(plan)} exercises ? {int(plan.duration_minutes.sum())+5} planned minutes</b>'
                     ' including a 5-minute preparation buffer. Exercise blocks include practice, rest and transitions.</p>'))
        for index,row in plan.iterrows():
            steps = ''.join(f'<li>{escape(line)}</li>' for line in row.instructions.splitlines()) if row.instructions else ''
            details = f'<ol>{steps}</ol>' if steps else '<p>Detailed instructions are not available for this exercise.</p>'
            display(HTML(f'<div style="padding:16px;margin:10px 0;border:1px solid #8c9b9d;border-radius:10px">'
                f'<h3>{index+1}. {escape(row["name"])}</h3>'
                f'<p>{escape(row.muscle.title())} ? {escape(row.required_equipment.replace("|",", "))} ? '
                f'<b>{row.duration_minutes} minutes</b></p>'
                f'<p>{escape(row.suggested_format)}</p>'
                f'<details><summary>Instructions</summary><p>{escape(row.description)}</p>{details}</details>'
                f'<details><summary>Why this exercise?</summary><p>{escape(row.reason)}</p></details></div>'))
        export = plan[['name','muscle','required_equipment','duration_minutes','suggested_format','instructions','reason']]
        display(HTML(download_link('Download my workout',export.to_csv(index=False),'my_workout.csv')+' &nbsp; | &nbsp; '+
                     download_link('Download my profile',json.dumps(asdict(profile),indent=2),'my_profile.json','application/json')))


def on_build(_):
    try:
        profile = read_profile()
        history = activities()
        model = Recommender(catalogue,history)
        ranked,meta = model.recommend(profile,top_n=30,exclude_seen=new_only.value)
        plan = make_plan(ranked,profile)
        save_profile(profile)
        state.update(profile=profile,plan=plan,recommendations=ranked)
        saved_profiles.options=['New profile']+list(profiles())
        saved_profiles.value=profile.user_id
        if not plan.empty:
            exercise.value=plan.iloc[0].exercise_id
            duration.value=float(plan.iloc[0].duration_minutes)
        log_button.disabled=False
        status.value=f'<p>Profile saved. Showing the workout for <b>{escape(profile.user_id)}</b>.</p>'
        feedback_status.value=''
        render_workout(profile,plan,meta)
        render_history()
    except (ValueError,KeyError) as exc:
        status.value=f'<p style="color:#c33">{escape(exc)}</p>'


def on_load(_):
    data=profiles().get(saved_profiles.value)
    if data is None:
        status.value='<p>Select a saved profile first.</p>'
        return
    user_id.value=data['user_id'];age.value=data['age'];gender.value=data['gender']
    height.value=data['height_cm'];weight.value=data['weight_kg'];goal.value=data['goal']
    experience.value=data['experience'];minutes.value=data['minutes']
    location.value=data['location'];energy.value=data['energy']
    equipment.value=tuple(v for v in data['equipment'] if v in available)
    focus.value=tuple(data['muscles']);exclude.value=tuple(data['excluded_muscles'])
    on_build(None)


def on_log(_):
    active=state['profile']
    if active is None:
        feedback_status.value='<p>Build your workout first.</p>'
        return
    try:
        log_activity(active.user_id,exercise.value,duration.value,completion.value/100,rating.value)
        feedback_status.value=(f'<p>Saved feedback for <b>{escape(exercise_names[exercise.value])}</b> '
            f'to <b>{escape(active.user_id)}</b>. Build again to use your updated preferences.</p>')
        render_history()
    except ValueError as exc:
        feedback_status.value=f'<p style="color:#c33">{escape(exc)}</p>'


def render_library(_=None):
    with library_view:
        clear_output(wait=True)
        found=catalogue[catalogue.name.str.contains(search.value,case=False,regex=False)]
        display(HTML(f'<p>{len(found)} exercises found. Showing up to 50.</p>'))
        display(found[['name','muscle','equipment','level']].head(50).rename(columns={
            'name':'Exercise','muscle':'Muscle','equipment':'Equipment','level':'Difficulty'}))

build_button.on_click(on_build)
load_button.on_click(on_load)
log_button.on_click(on_log)
search.observe(render_library,names='value')

profile_panel=w.VBox([
    w.HTML('<h2>Your profile</h2><p>Enter your details, select the equipment you have, then build your workout.</p>'),
    w.HBox([saved_profiles,load_button]),
    w.HBox([w.VBox([user_id,goal,experience,age,gender,height,weight]),w.VBox([minutes,location,energy,equipment])],layout=w.Layout(flex_flow='row wrap')),
    w.HTML('<p>Use Ctrl/Cmd-click for multiple selections or to clear selections. Empty focus means full body. Bodyweight is always included.</p>'),
    w.HBox([focus,exclude],layout=w.Layout(flex_flow='row wrap')),new_only,build_button,status,workout_view])
feedback_panel=w.VBox([w.HTML('<h2>Track your workout</h2><p>Feedback is recorded for the profile shown in your workout, even if you edit the input fields.</p>'),
    exercise,duration,completion,rating,log_button,feedback_status,history_view])
library_panel=w.VBox([w.HTML('<h2>Find an exercise</h2>'),search,library_view])
tabs=w.Tab(children=[profile_panel,feedback_panel,library_panel])
for i,title in enumerate(['My workout','Activity & feedback','Exercise library']): tabs.set_title(i,title)
display(w.HTML('<h1>Gym Workout Recommender</h1><p>Your goal. Your equipment. Your workout.</p>'))
display(tabs)
display(w.HTML('<p style="font-size:12px">Local adult fitness prototype. Duration blocks are planning estimates. Equipment metadata may be incomplete; muscle exclusions check primary muscles only. '
              'Exercise calories are not available. Your profiles and feedback stay in data/notebook_workouts.sqlite3.</p>'))
with workout_view: display(HTML('<p>Your personalized workout will appear here after you select <b>Build my workout</b>.</p>'))
render_history()
render_library()
